# AbstractDownload: PubMed Abstract Retrieval for UK Biobank

"
"This notebook downloads UK Biobank-related PubMed records (metadata + abstracts)
"
"using NCBI E-utilities (`esearch` + `efetch`) and writes a clean CSV for downstream analysis.

"
"Primary output:
"
"- `pubmed_abstract_updated.csv`

"
"Columns:
"
"- `Journal`, `Title`, `Abstract`, `Authors`, `Year`, `PMID`, `DOI`, `query_year_range`


## Conceptual pipeline

"
"1. Build the PubMed query for UK Biobank publications.
"
"2. For each year-range block, run `esearch` with `usehistory=y` to get `WebEnv` + `QueryKey`.
"
"3. Use `efetch` in batches (`retmax`) to download full PubMed XML records.
"
"4. Parse journal/title/abstract/authors/year/PMID/DOI from each `PubmedArticle`.
"
"5. Concatenate all blocks and save to `pubmed_abstract_updated.csv` (with optional PMID-based deduplication).


In [6]:
from pathlib import Path
import re
import time
import xml.etree.ElementTree as ET

import pandas as pd
import requests


try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


In [7]:
# -------------------------
# Configuration
# -------------------------
PROJECT_ROOT = Path('.').resolve()
OUTPUT_CSV = PROJECT_ROOT / 'pubmed_abstract_updated.csv'

NCBI_API_KEY = ''  # Optional but recommended for higher rate limits.

QUERY_BASE = '(UK Biobank[Title/Abstract]) OR (UK Biobank[Affiliation])'
YEAR_RANGES = [
    '2000:2009',
    '2010:2014',
    '2015:2019',
    '2020:2022',
    '2022:2024',
]

RETMAX = 500
SLEEP_SECONDS = 0.34
DEDUP_BY_PMID = True

print(f'Output CSV: {OUTPUT_CSV}')


Output CSV: /Users/ai421/Desktop/GitHub/UKBB_litreview_parkinsons/Parkinsons/pubmed_abstract_updated.csv


In [8]:
BASE_URL = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils'


def _safe_text(node):
    if node is None:
        return None
    text = ''.join(node.itertext()).strip()
    return text or None


def _extract_year(article) -> str | None:
    # Priority: ArticleDate/Year -> PubDate/Year -> PubDate/MedlineDate year regex
    for xpath in [
        './/ArticleDate/Year',
        './/JournalIssue/PubDate/Year',
        './/PubDate/Year',
    ]:
        node = article.find(xpath)
        value = _safe_text(node)
        if value and value.isdigit():
            return value

    medline_date = _safe_text(article.find('.//JournalIssue/PubDate/MedlineDate'))
    if medline_date:
        match = re.search(r'(19|20)\d{2}', medline_date)
        if match:
            return match.group(0)
    return None


def _extract_abstract(article) -> str | None:
    abstract_nodes = article.findall('.//Abstract/AbstractText')
    if not abstract_nodes:
        return None

    parts = []
    for node in abstract_nodes:
        label = node.attrib.get('Label')
        text = _safe_text(node)
        if not text:
            continue
        if label:
            parts.append(f'{label}: {text}')
        else:
            parts.append(text)

    abstract = ' '.join(parts).strip()
    return abstract or None


def _extract_doi(article) -> str | None:
    for node in article.findall('.//ArticleIdList/ArticleId'):
        if (node.attrib.get('IdType') or '').lower() == 'doi':
            doi = _safe_text(node)
            if doi:
                return doi

    for node in article.findall('.//ELocationID'):
        if (node.attrib.get('EIdType') or '').lower() == 'doi':
            doi = _safe_text(node)
            if doi:
                return doi

    return None


def _extract_authors(article) -> str | None:
    names = []
    for author in article.findall('.//AuthorList/Author'):
        last_name = _safe_text(author.find('LastName')) or ''
        fore_name = _safe_text(author.find('ForeName')) or ''
        collective = _safe_text(author.find('CollectiveName'))

        if collective:
            names.append(collective)
            continue

        full_name = f'{fore_name} {last_name}'.strip()
        if full_name:
            names.append(full_name)

    return ', '.join(names) if names else None


def fetch_pubmed_ukbb(
    query_base: str,
    year_ranges: list[str],
    api_key: str = '',
    retmax: int = 500,
    sleep_seconds: float = 0.34,
    dedup_by_pmid: bool = True,
) -> pd.DataFrame:
    session = requests.Session()
    records = []

    outer_iter = year_ranges
    if tqdm is not None:
        outer_iter = tqdm(year_ranges, desc='Year ranges', unit='range')

    for year_range in outer_iter:
        term = f'{query_base} AND ({year_range}[Publication Date])'
        search_params = {
            'db': 'pubmed',
            'term': term,
            'usehistory': 'y',
            'retmode': 'xml',
        }
        if api_key.strip():
            search_params['api_key'] = api_key.strip()

        search_response = session.get(f'{BASE_URL}/esearch.fcgi', params=search_params, timeout=60)
        search_response.raise_for_status()

        search_root = ET.fromstring(search_response.text)
        total_count = int((_safe_text(search_root.find('Count')) or '0'))
        webenv = _safe_text(search_root.find('WebEnv'))
        query_key = _safe_text(search_root.find('QueryKey'))

        if not webenv or not query_key:
            print(f'[WARN] Missing WebEnv/QueryKey for {year_range}; skipping this block.')
            continue

        print(f'Year range {year_range}: {total_count} records')

        progress = None
        if tqdm is not None:
            progress = tqdm(total=total_count, desc=f'Fetching {year_range}', unit='records', leave=False)

        retstart = 0
        while retstart < total_count:
            fetch_params = {
                'db': 'pubmed',
                'query_key': query_key,
                'WebEnv': webenv,
                'retstart': retstart,
                'retmax': retmax,
                'retmode': 'xml',
                'rettype': 'abstract',
            }
            if api_key.strip():
                fetch_params['api_key'] = api_key.strip()

            fetch_response = session.get(f'{BASE_URL}/efetch.fcgi', params=fetch_params, timeout=90)
            fetch_response.raise_for_status()
            fetch_root = ET.fromstring(fetch_response.text)

            batch_articles = fetch_root.findall('.//PubmedArticle')
            for article in batch_articles:
                record = {
                    'Journal': _safe_text(article.find('.//Journal/Title')),
                    'Title': _safe_text(article.find('.//ArticleTitle')),
                    'Abstract': _extract_abstract(article),
                    'Authors': _extract_authors(article),
                    'Year': _extract_year(article),
                    'PMID': _safe_text(article.find('.//PMID')),
                    'DOI': _extract_doi(article),
                    'query_year_range': year_range,
                }
                records.append(record)

            if progress is not None:
                progress.update(len(batch_articles))

            retstart += retmax
            if sleep_seconds > 0:
                time.sleep(sleep_seconds)

        if progress is not None:
            progress.close()

    df = pd.DataFrame.from_records(records)
    if df.empty:
        return df

    df['Year'] = pd.to_numeric(df['Year'], errors='coerce').astype('Int64')

    if dedup_by_pmid and 'PMID' in df.columns:
        non_null = df['PMID'].notna() & (df['PMID'].astype(str).str.strip() != '')
        df_non_null = df[non_null].drop_duplicates(subset=['PMID'], keep='first')
        df_null = df[~non_null]
        df = pd.concat([df_non_null, df_null], ignore_index=True)

    return df


In [9]:
pubmed_df = fetch_pubmed_ukbb(
    query_base=QUERY_BASE,
    year_ranges=YEAR_RANGES,
    api_key=NCBI_API_KEY,
    retmax=RETMAX,
    sleep_seconds=SLEEP_SECONDS,
    dedup_by_pmid=DEDUP_BY_PMID,
)

if pubmed_df.empty:
    raise RuntimeError('No PubMed records were retrieved. Check query/API/network settings.')

pubmed_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')

print(f'Saved {len(pubmed_df):,} records to {OUTPUT_CSV}')
pubmed_df.head()


Year range 2000:2009: 38 records
Year range 2010:2014: 41 records
Year range 2015:2019: 1037 records
Year range 2020:2022: 3687 records
Year range 2022:2024: 7158 records
Saved 10,100 records to /Users/ai421/Desktop/GitHub/UKBB_litreview_parkinsons/Parkinsons/pubmed_abstract_updated.csv


,Journal,Title,Abstract,Authors,Year,PMID,DOI,query_year_range
0,The Health service journal,Population health data. Get a load of me.,"UK Biobank has around 330,000 people signed up...",Mark Gould,2009,20726090,NaN,2000:2009
1,"Lancet (London, England)",Role of the UK Biobank Ethics and Governance C...,NaN,Graeme Laurie,2009,19914512,10.1016/S0140-6736(09)61989-9,2000:2009
2,"International journal of surgery (London, Engl...",The UK Biobank project: trust and altruism are...,NaN,Hazel Thornton,2009,19748601,10.1016/j.ijsu.2009.09.001,2000:2009
3,"Lancet (London, England)",An afternoon at UK Biobank.,NaN,NaN,2009,19345813,10.1016/S0140-6736(09)60664-4,2000:2009
4,Mutagenesis,Environmental exposure measurement in cancer e...,"Environmental exposures, used in the broadest ...",Christopher P Wild,2008,19033256,10.1093/mutage/gen061,2000:2009


In [10]:
summary = {
    'rows': len(pubmed_df),
    'missing_abstracts': int(pubmed_df['Abstract'].isna().sum()) if 'Abstract' in pubmed_df.columns else None,
    'missing_doi': int(pubmed_df['DOI'].isna().sum()) if 'DOI' in pubmed_df.columns else None,
    'year_min': int(pubmed_df['Year'].dropna().min()) if 'Year' in pubmed_df.columns and pubmed_df['Year'].dropna().shape[0] else None,
    'year_max': int(pubmed_df['Year'].dropna().max()) if 'Year' in pubmed_df.columns and pubmed_df['Year'].dropna().shape[0] else None,
}
summary


{'rows': 10100,
 'missing_abstracts': 268,
 'missing_doi': 31,
 'year_min': 2003,
 'year_max': 2025}